In [1]:
import os 
os.chdir('../../')

In [2]:
from backbones.sana import SANA

# 사용 예
model = SANA()
print(model)

/home/scpark/miniconda3/envs/rbf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading pipeline components...: 100%|██████████| 5/5 [00:01<00:00,  2.83it/s]


In [3]:
import matplotlib.pyplot as plt

def show_compare(euler_img, dpm_img, unipc_img):
    fig, axes = plt.subplots(1, 3, figsize=(18, 10))
    axes[0].imshow(euler_img); axes[0].axis('off'); axes[0].set_title('Euler')
    axes[1].imshow(dpm_img);  axes[1].axis('off'); axes[1].set_title('DPM-Solver')
    axes[2].imshow(unipc_img);  axes[2].axis('off'); axes[2].set_title('UniPC')
    plt.tight_layout()
    plt.show()


In [4]:
from solvers.others.euler_solver import Euler_Solver

pos_texts = ["A serene twilight view of a traditional Korean hanok village nestled between misty mountain slopes: curved midnight-blue tiled eaves, softly glowing paper lanterns swaying in the breeze; ancient pine trees arching over stone pathways; intricate wooden lattice windows casting delicate shadows; a lone scholar in flowing hanbok practicing calligraphy beside a koi pond with lotus petals drifting on the water; cinematic 8K ultra-realism with dynamic volumetric moonlight filtering through morning mist; painterly strokes blending classical Joseon-era ink wash with modern hyperrealism; shot on RED Monstro 8K, 50 mm f/1.2 lens; subtle film grain; maximum fidelity; emotional atmosphere."]
neg_texts = ["lowres, bad anatomy, deformed, blurry, pixelated, oversaturated, underexposed, overexposed, artifact, jpeg artifacts, watermark, text, logo, extra limbs, mutated hands, unnatural colors, noisy background, out of focus, poor composition, cultural clichés, stereotype exaggeration, flat lighting, glitch"]

noise_schedule = model.get_noise_schedule()
model_fn = model.get_model_fn(noise_schedule=noise_schedule, pos_conds=pos_texts, neg_conds=neg_texts, guidance_scale=4.5)
latents = model.get_noise(seeds=[42])

NFE = 10
solver = Euler_Solver(noise_schedule, steps=NFE, skip_type='time_uniform_flow', flow_shift=3.0, algorithm_type="data_prediction")
euler_latents = solver.sample(latents, model_fn)['samples']
euler_samples = model.decode_vae(euler_latents, raw_output=True)
euler_samples


tensor([[[[-0.3672, -0.3066, -0.3164,  ..., -0.9102, -0.8828, -0.8047],
          [-0.3438, -0.3574, -0.3574,  ..., -0.8672, -0.9297, -0.8750],
          [-0.3555, -0.3516, -0.3555,  ..., -0.9375, -0.9375, -0.9258],
          ...,
          [-0.9844, -0.9375, -0.9688,  ..., -0.8164, -0.8867, -0.8672],
          [-1.0000, -0.9453, -0.8984,  ..., -0.9375, -0.7383, -0.8320],
          [-0.9414, -0.9766, -0.9648,  ..., -0.8711, -0.7617, -0.8242]],

         [[-0.0908, -0.0081,  0.0153,  ..., -0.8242, -0.8047, -0.7344],
          [-0.0233, -0.0479, -0.0281,  ..., -0.7344, -0.7969, -0.8047],
          [-0.0298, -0.0449, -0.0300,  ..., -0.8125, -0.8438, -0.8633],
          ...,
          [-0.8750, -0.8906, -0.9141,  ..., -0.6289, -0.7539, -0.7148],
          [-0.8945, -0.8750, -0.8047,  ..., -0.8281, -0.4707, -0.6406],
          [-0.8438, -0.8789, -0.8867,  ..., -0.7500, -0.5742, -0.6680]],

         [[ 0.2734,  0.3691,  0.3848,  ..., -0.7852, -0.7500, -0.7656],
          [ 0.3770,  0.3203,  

In [5]:
euler_samples.shape

torch.Size([1, 3, 1024, 1024])